In [1]:
from dotenv import load_dotenv
import os
from pathlib import Path

#root_path = Path().resolve().parent

load_dotenv(".env")

TMDB_TOKEN = os.getenv("TMDB_TOKEN")


In [2]:
import os
import time
import requests
import pandas as pd

# ── Konfiguration ─────────────────────────────────────

TOTAL_MOVIES = 1600
PAGES = 80

OUTPUT_FILE = os.path.join(os.getcwd(), "Q6.csv")

tmdb_headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_TOKEN}"
}

all_rows = []


# ── Step 1: Filme laden ───────────────────────────────

movies = []

for page in range(1, PAGES + 1):

    resp = requests.get(
        "https://api.themoviedb.org/3/discover/movie",
        headers=tmdb_headers,
        params={
            "sort_by": "popularity.desc",
            "vote_count.gte": 500,
            "primary_release_date.gte": "2010-01-01",
            "primary_release_date.lte": "2023-12-31",
            "page": page
        }
    ).json()

    movies.extend(resp["results"])

movies = movies[:TOTAL_MOVIES]


# ── Step 2: Details laden ─────────────────────────────

for movie in movies:

    details = requests.get(
        f"https://api.themoviedb.org/3/movie/{movie['id']}",
        headers=tmdb_headers
    ).json()

    collection = details.get("belongs_to_collection")

    if collection is None:
        continue

    budget = details.get("budget")
    revenue = details.get("revenue")

    if not budget or not revenue:
        continue

    all_rows.append({
        "collection_id": collection["id"],
        "collection_name": collection["name"],
        "title": details.get("title"),
        "release_date": details.get("release_date"),
        "budget": budget,
        "revenue": revenue,
        "roi": round(revenue / budget, 2),
        "vote_average": details.get("vote_average"),
        "vote_count": details.get("vote_count")
    })

    print("Loaded:", details.get("title"))

    time.sleep(0.3)


df = pd.DataFrame(all_rows)

df["release_date"] = pd.to_datetime(df["release_date"])


# ── Step 3: Originalfilm pro Franchise bestimmen ─────

df = df.sort_values(["collection_id", "release_date"])

df["is_original"] = df.groupby("collection_id").cumcount() == 0


# ── Step 4: Vergleichswerte berechnen ─────────────────

originals = df[df["is_original"]][[
    "collection_id",
    "revenue",
    "roi",
    "vote_average"
]].rename(columns={
    "revenue": "original_revenue",
    "roi": "original_roi",
    "vote_average": "original_rating"
})

df = df.merge(originals, on="collection_id")

df["revenue_diff"] = df["revenue"] - df["original_revenue"]
df["roi_diff"] = df["roi"] - df["original_roi"]
df["rating_diff"] = df["vote_average"] - df["original_rating"]


# ── Step 5: CSV speichern ────────────────────────────

df.to_csv(OUTPUT_FILE, index=False)

print("\nDataset erstellt")
print("Filme:", len(df))
print("CSV:", OUTPUT_FILE)

print(df.head())

Loaded: The Avengers
Loaded: Scream VI
Loaded: Dune
Loaded: Scream
Loaded: Scream 4
Loaded: Zootopia
Loaded: Fast X
Loaded: Toy Story 4
Loaded: The Equalizer
Loaded: Avatar: The Way of Water
Loaded: Avengers: Infinity War
Loaded: The Batman
Loaded: Joker
Loaded: The Hobbit: The Battle of the Five Armies
Loaded: Spider-Man: No Way Home
Loaded: Maleficent
Loaded: Spider-Man: Across the Spider-Verse
Loaded: Coco
Loaded: Frozen
Loaded: Tangled
Loaded: Greenland
Loaded: Stake Land
Loaded: Top Gun: Maverick
Loaded: Terminator Genisys
Loaded: Mortal Kombat
Loaded: Pacific Rim
Loaded: Guardians of the Galaxy Vol. 2
Loaded: Wonka
Loaded: Spider-Man: Into the Spider-Verse
Loaded: The Super Mario Bros. Movie
Loaded: Aquaman
Loaded: Venom
Loaded: Jurassic World
Loaded: Scary Movie 5
Loaded: Spider-Man: Homecoming
Loaded: The Dark Knight Rises
Loaded: Expend4bles
Loaded: Pitch Perfect
Loaded: Godzilla
Loaded: Sicario
Loaded: Once Upon a Time... in Hollywood
Loaded: Skyfall
Loaded: Terrifier
Loaded: